# Embedding Generation


1. `all-MiniLM-L6-v2` 
2. `nomic-ai/nomic-embed-text-v1` 
3. `text-embedding-3-small` 


In [ ]:
!pip install sentence-transformers openai

In [ ]:
import json
import os
import time
import numpy as np
from typing import List, Dict, Any

In [ ]:
# Config
INPUT_PATH = "data/chunks_fixed300_overlap20pct.jsonl" 
OUTPUT_DIR = "data/"
BATCH_SIZE = 16

# OpenAI config 
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

## Load Chunks

In [ ]:
def load_chunks(path: str) -> List[Dict[str, Any]]:
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))
    print(f"Loaded {len(chunks)} chunks from {path}")
    return chunks

chunks = load_chunks(INPUT_PATH)
texts = [c["text"] for c in chunks]

In [ ]:
def save_embedded_chunks(
    chunks: List[Dict],
    embeddings: np.ndarray,
    output_path: str,
):
    """Write chunks with embeddings filled in to JSONL."""
    with open(output_path, "w", encoding="utf-8") as f:
        for chunk, emb in zip(chunks, embeddings):
            record = {
                "chunk_id": chunk["chunk_id"],
                "text": chunk["text"],
                "embedding": emb.tolist(),
                "metadata": chunk["metadata"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"Saved to {output_path} ({size_mb:.2f} MB)")

## Model 1: all-MiniLM-L6-v2

Lightweight sentence-transformer (22M params, 384d). Fast and widely used as a baseline.

In [ ]:
from sentence_transformers import SentenceTransformer

model_minilm = SentenceTransformer("all-MiniLM-L6-v2")

t0 = time.time()
embeddings_minilm = model_minilm.encode(
    texts, batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True
)
elapsed = time.time() - t0

print(f"Shape: {embeddings_minilm.shape}")
print(f"Time : {elapsed:.1f}s ({len(texts)/elapsed:.0f} chunks/s)")

save_embedded_chunks(
    chunks, embeddings_minilm,
    os.path.join(OUTPUT_DIR, "embedded_minilm.jsonl")
)

## Model 2: nomic-ai/nomic-embed-text-v1

Open-source 768d model with strong MTEB scores. Requires a task prefix for documents.

In [ ]:
model_nomic = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1", trust_remote_code=True
)

# nomic requires prefix: "search_document: " for indexing, "search_query: " for queries
texts_prefixed = [f"search_document: {t}" for t in texts]

t0 = time.time()
embeddings_nomic = model_nomic.encode(
    texts_prefixed, batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True
)
elapsed = time.time() - t0

print(f"Shape: {embeddings_nomic.shape}")
print(f"Time : {elapsed:.1f}s ({len(texts)/elapsed:.0f} chunks/s)")

save_embedded_chunks(
    chunks, embeddings_nomic,
    os.path.join(OUTPUT_DIR, "embedded_nomic.jsonl")
)

## Model 3: text-embedding-3-small (OpenAI API)

OpenAI's small embedding model (1536d). Requires an API key. Batched to respect rate limits.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
OPENAI_MODEL = "text-embedding-3-small"
OPENAI_BATCH = 100 


def embed_openai(texts: List[str], model: str = OPENAI_MODEL, batch_size: int = OPENAI_BATCH) -> np.ndarray:
    """Call OpenAI embedding API in batches."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(input=batch, model=model)
        batch_embs = [item.embedding for item in resp.data]
        all_embs.extend(batch_embs)
        if i + batch_size < len(texts):
            time.sleep(0.5)  # basic rate-limit backoff
    return np.array(all_embs, dtype=np.float32)


t0 = time.time()
embeddings_openai = embed_openai(texts)
elapsed = time.time() - t0

print(f"Shape: {embeddings_openai.shape}")
print(f"Time : {elapsed:.1f}s ({len(texts)/elapsed:.0f} chunks/s)")

save_embedded_chunks(
    chunks, embeddings_openai,
    os.path.join(OUTPUT_DIR, "embedded_openai_small.jsonl")
)

## Summary

In [ ]:
print(f"{'Model':35s} {'Dim':>6s} {'File':>30s}")
print("-" * 75)
for name, emb, fname in [
    ("all-MiniLM-L6-v2", embeddings_minilm, "embedded_minilm.jsonl"),
    ("nomic-embed-text-v1", embeddings_nomic, "embedded_nomic.jsonl"),
    ("text-embedding-3-small", embeddings_openai, "embedded_openai_small.jsonl"),
]:
    path = os.path.join(OUTPUT_DIR, fname)
    size = os.path.getsize(path) / (1024 * 1024)
    print(f"{name:35s} {emb.shape[1]:>6d} {fname:>30s} ({size:.1f} MB)")

In [ ]:
# Verify: load first record from each file
for fname in ["embedded_minilm.jsonl", "embedded_nomic.jsonl", "embedded_openai_small.jsonl"]:
    path = os.path.join(OUTPUT_DIR, fname)
    with open(path, "r", encoding="utf-8") as f:
        rec = json.loads(f.readline())
    emb_len = len(rec["embedding"])
    print(f"[{fname}] chunk_id={rec['chunk_id']}  embedding_dim={emb_len}")